In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import sys 
import torch
from pathlib import Path

from satchange import (
    BCEDiceLoss,
    BinaryChangeMetrics,
)

from satchange.models import FCSiamDiff

from torch.utils.data import DataLoader
from satchange.utils import load_config
from satchange.data import (
    RandomFlipRotate,
    create_dataset_dataloader,
    prepare_levir_cd,
)
from satchange.experiments import (
    load_fc_siam_diff_experiment
)

In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


In [3]:
print("Python executable:", sys.executable)
print("Python version:", sys.version)

print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("Visible GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("Selected GPU:", torch.cuda.get_device_name(0))

Python executable: .venv/bin/python
Python version: 3.11.6 | packaged by conda-forge | (main, Oct  3 2023, 10:40:35) [GCC 12.3.0]
PyTorch: 2.12.1+cu130
PyTorch CUDA: 13.0
CUDA available: True
Visible GPU count: 1
Selected GPU: NVIDIA A100-SXM4-80GB


In [4]:
current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

full_config_path = (
    project_root
    / "configs"
    / "fc_siam_diff_full.yaml"
)

resized_config_path = (
    project_root
    / "configs"
    / "fc_siam_diff_resized.yaml"
)


print("The project root: ", project_root)

print("Full config:", full_config_path)
print("Full config exists:", full_config_path.exists())

print("Resized config:", resized_config_path)
print("Resized config exists:", resized_config_path.exists())

The project root:  .
Full config: configs/fc_siam_diff_full.yaml
Full config exists: True
Resized config: configs/fc_siam_diff_resized.yaml
Resized config exists: True


### Resized Dataset

In [5]:
EXPERIMENT = "resized"
# Available "full" and "resized"


config, config_path = load_fc_siam_diff_experiment(
    experiment=EXPERIMENT,
    project_root=project_root,
)

print("Experiment:", EXPERIMENT)
print("Configuration:", config_path)
print("Image size:", config["data"]["image_size"])
print("Batch size:", config["data"]["batch_size"])
print("Output directory:", config["output"]["directory"])

Experiment: resized
Configuration: configs/fc_siam_diff_resized.yaml
Image size: 256
Batch size: 16
Output directory: artifacts/fc_siam_diff_resized


In [6]:
# path to the dataset
dataset_root = prepare_levir_cd(
    config=config,
    project_root=project_root,
)

print("Ready dataset root:", dataset_root)

NameError: name 'prepare_levir_cd' is not defined

In [ ]:
train_transform = RandomFlipRotate(
    horizontal_flip_probability = 0.5,
    vertical_flip_probability = 0.5,
)

train_dataset, train_loader = create_levir_cd_split(
    dataset_root=dataset_root,
    split="train",
    image_size=config["data"]["image_size"],
    batch_size=config["data"]["batch_size"],
    num_workers=config["data"]["num_workers"],
    transform=train_transform,
    shuffle=True,
    pin_memory=torch.cuda.is_available(),
    seed=config["project"]["seed"],
)

validation_dataset, validation_loader = create_dataset_dataloader(
    dataset_root=dataset_root,
    split="val",
    image_size=config["data"]["image_size"],
    batch_size=config["data"]["batch_size"],
    num_workers=config["data"]["num_workers"],
    pin_memory=torch.cuda.is_available(),
)

test_dataset, test_loader = create_dataset_dataloader(
    dataset_root=dataset_root,
    split="test",
    image_size=config["data"]["image_size"],
    batch_size=config["data"]["batch_size"],
    num_workers=config["data"]["num_workers"],
    pin_memory=torch.cuda.is_available(),
)

In [ ]:
RANDOM_SEED = config["project"]["seed"]

torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        RANDOM_SEED
    )


model = FCSiamDiff(
    in_channels=(
        config["model"]["parameters"]["in_channels"]
    ),
    out_channels=(
        config["model"]["parameters"]["out_channels"]
    ),
    base_channels=(
        config["model"]["parameters"]["base_channels"]
    ),
).to(device)


loss_function = BCEDiceLoss(
    bce_weight=(
        config["training"]["loss"]["bce_weight"]
    ),
    dice_weight=(
        config["training"]["loss"]["dice_weight"]
    ),
    pos_weight=5,
).to(device)


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config["training"]["learning_rate"],
    weight_decay=(
        config["training"]["weight_decay"]
    ),
)

BCE positive weight: tensor([[[20.7911]]], device='cuda:0')


In [ ]:
THRESHOLDS = [
    round(value / 100, 2)
    for value in range(5, 96, 5)
]

batch_size = config["data"]["batch_size"]
learning_rate = config["training"]["learning_rate"]
seed = RANDOM_SEED
positive_weight = 10

MODEL_NAME = "fc_siam_diff"

RUN_NAME = (
    f"levir_{EXPERIMENT}_"
    f"pw{positive_weight:g}_"
    f"lr{learning_rate:g}_"
    f"bs{batch_size}_"
    f"seed{seed}"
)


# Reset random initialization for a clean experiment.
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)


# Create a new model with fresh initial weights.
model = FCSiamDiff(
    in_channels=(
        config["model"]["parameters"]["in_channels"]
    ),
    out_channels=(
        config["model"]["parameters"]["out_channels"]
    ),
    base_channels=(
        config["model"]["parameters"]["base_channels"]
    ),
).to(device)


# Create a new loss function using the selected pos_weight.
loss_function = BCEDiceLoss(
    bce_weight=(
        config["training"]["loss"]["bce_weight"]
    ),
    dice_weight=(
        config["training"]["loss"]["dice_weight"]
    ),
    pos_weight=positive_weight,
).to(device)


# Create a new optimizer with no previous AdamW history.
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=(
        config["training"]["weight_decay"]
    ),
)


print("Model name:", MODEL_NAME)
print("Run name:", RUN_NAME)
print("Number of epochs:", 80)
print("Positive weight:", positive_weight)
print("Learning rate:", learning_rate)
print("Batch size:", batch_size)
print("Seed:", seed)


history = fit_binary_change_model(
    model=model,
    train_loader=train_loader,
    validation_loader=validation_loader,
    loss_function=loss_function,
    optimizer=optimizer,
    device=device,
    number_of_epochs=80,
    thresholds=THRESHOLDS,
    output_root=(
        project_root
        / "artifacts"
        / "training_runs"
    ),
    model_name=MODEL_NAME,
    run_name=RUN_NAME,
    checkpoint_interval=5,
    config=config,
    resume_from=None,
)

### Full sized Dataset

In [ ]:
# EXPERIMENT = "full"

# config, config_path = load_fc_siam_diff_experiment(
#     experiment=EXPERIMENT,
#     project_root=project_root,
# )

# print(f"Configuration {config_path}")
# print(f"Data settings: {config['data']}")

In [ ]:
# dataset_root = prepare_levir_cd(
#     config=config,
#     project_root=project_root,
# )


# dataset = LEVIRCDDataset(
#     dataset_root=dataset_root,
#     split="train",
#     image_size=config["data"]["image_size"],
# )

# sample=dataset[0]


# print(f"Number of samples: {len(dataset)}")
# print(f"Sample ID: {sample['sample_id']}")
# print(f"Before shape: {sample['before'].shape}")
# print(f"After shape: {sample['after'].shape}")
# print(f"Mask shape: {sample['mask'].shape}")

In [ ]:
# before_features = model._encode(before)
# after_features = model._encode(after)

# feature_differences = model._compute_feature_differences(
#     before_features,
#     after_features,
# )

# decoded_features = model._decode(
#     feature_differences
# )

# print(decoded_features.shape)

In [ ]:
# # switches the model into training mode
# model.train()
# # and
# model.eval()

# for batch in train_loader:
#     # cleares gradients left from previous batches 'cause PyTorch accumulates gradients
#     optimizer.zero_grad()

#     # the prediction and computation graph
#     prediction = model(
#         batch["before"],
#         batch["after"],
#     )

#     # computing the loss (how different the prediction from the real mask)
#     loss = loss_function(
#         prediction,
#         batch["mask"],
#     )

#     # computing the gradient (what change to reduce the loss)
#     loss.backward()

#     # update the weights using gradients
#     optimizer.step()


# # PyTorch cycle that does not effect the model. It's used to evaluate but not to train.
# with torch.no_grad():
#     for batch in validation_loader:
#         prediction = model(
#             batch["before"],
#             batch["after"],
#         )

#         # computing the loss
#         loss = loss_function(
#             prediction,
#             batch["mask"]
#         )

### Full sized Dataset

In [ ]:
# EXPERIMENT = "full"

# config, config_path = load_fc_siam_diff_experiment(
#     experiment=EXPERIMENT,
#     project_root=project_root,
# )

# print(f"Configuration {config_path}")
# print(f"Data settings: {config['data']}")

In [ ]:
# dataset_root = prepare_levir_cd(
#     config=config,
#     project_root=project_root,
# )


# dataset = LEVIRCDDataset(
#     dataset_root=dataset_root,
#     split="train",
#     image_size=config["data"]["image_size"],
# )

# sample=dataset[0]


# print(f"Number of samples: {len(dataset)}")
# print(f"Sample ID: {sample['sample_id']}")
# print(f"Before shape: {sample['before'].shape}")
# print(f"After shape: {sample['after'].shape}")
# print(f"Mask shape: {sample['mask'].shape}")